## Set up the Phoenix connection and establish a test dataset
### Connects to Phoenix

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={os.getenv('PHOENIX_API_KEY')}"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com/s/linkedInArticleGen"

from phoenix.client import Client
client = Client()
print("Connected to Phoenix")

Connected to Phoenix


### Initiate new dataset (only needs running once when starting new dataset)
Read the transcript from Class 4 from a local file, then creates a dataset in Phoenix with that transcript stored as a re-usable test set. 

Later experiments will feed this test case for later tests for objective comparison of outputs.

In [ ]:
## Read the dataset
with open("Class_4_test_sample.txt", "r") as f:
    raw_material = f.read()

## Creates a dataset with the given name and stored experiment
dataset = client.datasets.create_dataset(
    name="initial_test_from_Class_4",
    dataset_description="Lecture transcripts and articles from class 4 for testing initial batch of prompts",
    examples=[
        {
            "input": {"raw_material": raw_material},
            "output": {},
            "metadata": {"source": "class 4 transcript"}
        }
    ]
)
print(f"Created dataset: {dataset.id}")

### Run old dataset

In [2]:
# Load existing dataset (already created in Phoenix)
dataset = client.datasets.get_dataset(dataset="initial_test_from_Class_4")
print(f"Loaded dataset: {dataset.id}")

Loaded dataset: RGF0YXNldDoy


# Run the experiment for angleProposalPrompt1.md
### Load the experiment and store the output

In [ ]:
# Load the Phoenix experiment runner
from phoenix.client.experiments import run_experiment
from datetime import datetime
import anthropic

# Helper to read prompt files from the prompts/ folder
def read_prompt(filename):
    with open(f"prompts/{filename}", "r") as f:
        return f.read()

# Load current angle proposal prompt (re-reads from file each time so edits are picked up)
angle_prompt = read_prompt("angleProposalPrompt1.md")

# Define the task — Phoenix runs this against each example in the dataset
def angle_proposal_task(input):
    api_client = anthropic.Anthropic()
    message = api_client.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=16000,
        system=angle_prompt,
        messages=[{"role": "user", "content": input["raw_material"]}],
    )
    return message.content[0].text

# Auto-generate unique experiment name from timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"angle-proposal-{timestamp}"

# Run the experiment
latest_experiment = run_experiment(
    dataset=dataset,
    task=angle_proposal_task,
    experiment_name=experiment_name,
    experiment_description="Auto-run",
    timeout=120
)

print(f"Experiment: {experiment_name}")

## Export the experiment in a legible format

### Export the prompt sys message

In [ ]:
# See what was sent to the API as the system prompt
print("=" * 60)
print("SYSTEM PROMPT:")
print("=" * 60)
print(angle_prompt)

### For exporting the user message

In [ ]:
# See the input that was sent as the user message
print("=" * 60)
print("USER MESSAGE (first 500 chars):")
print("=" * 60)
print(raw_material[:500])

### For exporting a single experiment output

In [ ]:
# Pull the latest experiment results in a readable format
for run in latest_experiment["task_runs"]:
    print("=" * 60)
    print("OUTPUT:")
    print("=" * 60)
    print(run["output"])
    print("\n\n")

# Run the experiment up to supervisorPrompt1.5 04.04.26
## Run the pipeline up to supervisorPrompt

In [ ]:
import time
import re
from datetime import datetime
from phoenix.client.experiments import run_experiment
import anthropic

def read_prompt(filename):
    with open(f"prompts/{filename}", "r") as f:
        return f.read()

def extract_final_pitches(output):
    """Extract only the Final Pitches section, stripping all reasoning."""
    marker = "## Final Pitches"
    idx = output.find(marker)
    if idx != -1:
        return output[idx + len(marker):].strip()
    match = re.search(r"##\s*Final\s*Pitches", output, re.IGNORECASE)
    if match:
        return output[match.end():].strip()
    return output.strip()

def full_angle_pipeline(input):
    api_client = anthropic.Anthropic()
    
    # Load prompts
    angle_prompt = read_prompt("angleProposalPrompt1.md")
    supervisor_prompt = read_prompt("supervisorPrompt1.5.md")
    
    # Run angle proposal 3 times, extract only final pitches
    extracted_pitches = []
    for i in range(1, 4):
        if i > 1:
            time.sleep(10)
        message = api_client.messages.create(
            model="claude-opus-4-5-20251101",
            max_tokens=16000,
            system=angle_prompt,
            messages=[{"role": "user", "content": input["raw_material"]}],
        )
        pitches = extract_final_pitches(message.content[0].text)
        extracted_pitches.append(pitches)
    
    # Combine extracted pitches into labeled block
    combined = ""
    for i, pitches in enumerate(extracted_pitches, 1):
        combined += f"## Batch {i}\n\n{pitches}\n\n"
    
    # Run supervisor on combined pitches
    supervisor_message = api_client.messages.create(
        model="claude-opus-4-5-20251101",
        max_tokens=16000,
        system=supervisor_prompt,
        messages=[{"role": "user", "content": combined}],
    )
    
    # Return both what supervisor received and what it chose
    return {
        "supervisor_input": combined,
        "supervisor_output": supervisor_message.content[0].text
    }

# ---- CHANGE THIS EACH RUN ----
run_description = "v1.5 - updated supervisor prompt"
# --------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
experiment_name = f"full-pipeline-upToSupervisor-{timestamp}"

latest_experiment = run_experiment(
    dataset=dataset,
    task=full_angle_pipeline,
    experiment_name=experiment_name,
    experiment_description=run_description,
    timeout=1000
)

print(f"Experiment: {experiment_name}")
print(f"Description: {run_description}")

🧪 Experiment started.
📺 View dataset experiments: https://app.phoenix.arize.com/s/linkedInArticleGen/datasets/RGF0YXNldDoy/experiments
🔗 View this experiment: https://app.phoenix.arize.com/s/linkedInArticleGen/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDoxNA==


running tasks |          | 0/1 (0.0%) | ⏳ 00:00<? | ?it/s

In [ ]:
for run in latest_experiment["task_runs"]:
    output = run["output"]
    
    print("=" * 60)
    print("WHAT THE SUPERVISOR RECEIVED:")
    print("=" * 60)
    print(output["supervisor_input"])
    
    print("\n\n")
    
    print("=" * 60)
    print("WHAT THE SUPERVISOR CHOSE:")
    print("=" * 60)
    print(output["supervisor_output"])

In [ ]:
dir(client.experiments)

In [ ]:
help(client.experiments.get_experiment)

In [ ]:
experiments = client.experiments.list(dataset_id=dataset.id)
for exp in experiments:
    print(exp)

In [ ]:
failed_ids = [
    "RXhwZXJpbWVudDoxMg==",
    "RXhwZXJpbWVudDoxMQ==",
    "RXhwZXJpbWVudDoxMA==",
    "RXhwZXJpbWVudDo5",
    "RXhwZXJpbWVudDo4",
    "RXhwZXJpbWVudDo3",
    "RXhwZXJpbWVudDo2",
    "RXhwZXJpbWVudDoz",
]

for exp_id in failed_ids:
    try:
        client.experiments.delete(experiment_id=exp_id)
        print(f"Deleted: {exp_id}")
    except Exception as e:
        print(f"Failed to delete {exp_id}: {e}")

In [ ]:
successful_ids = [
    "RXhwZXJpbWVudDoxMw==",  # 12:34:45
    "RXhwZXJpbWVudDo1",      # 12:28:52
    "RXhwZXJpbWVudDo0",      # 11:21:06
]

for i, exp_id in enumerate(successful_ids, 1):
    print("=" * 80)
    print(f"RUN {i}")
    print("=" * 80)
    try:
        exp = client.experiments.get_experiment(experiment_id=exp_id)
        for run in exp["task_runs"]:
            output = run["output"]
            print("\n--- SUPERVISOR INPUT ---")
            print(output["supervisor_input"])
            print("\n--- SUPERVISOR OUTPUT ---")
            print(output["supervisor_output"])
    except Exception as e:
        print(f"Error: {e}")
    print("\n\n")